# Grandmaster Model: Die 0.82er Strategie

Wenn andere Leute 0.82 haben, dann nutzen sie das volle Potenzial der Datenstruktur aus!
Unsere Daten haben die Form `(300, 6)`. Das sind nicht einfach 6 zufällige Kanäle, sondern in der Regel:
*   **Kanal 0, 1, 2:** Accelerometer (Beschleunigung in X, Y, Z)
    
*   **Kanal 3, 4, 5:** Gyroscope (Drehrate in X, Y, Z)

Bisher haben wir die Magnitude (Stärke) und Korrelationen nur für die ersten drei Kanäle berechnet. 
Jetzt trennen wir das sauber: Wir berechnen die Magnitude für Beschleunigung UND Drehung. 
Zusätzlich berechnen wir die Korrelation zwischen der Beschleunigung und der Drehung!

Außerdem machen wir unser Zeit-Gitter feiner: Statt 5 Blöcke à 60 Sekunden schneiden wir die Daten jetzt in **10 Blöcke à 30 Sekunden**, damit LightGBM extrem feine Bewegungsmuster erkennt.

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

## 1. Daten laden

In [ ]:
KAGGLE_PATH = '/kaggle/input/datasets/axxtur/nycu-data-mining-assignment-3'
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'
    if not os.path.exists(KAGGLE_PATH):
        KAGGLE_PATH = 'nycu-data-mining-assignment-3'

train_data = np.load(os.path.join(KAGGLE_PATH, 'train_data.npz'), allow_pickle=True)
X_train_raw = train_data['X']
y_train = train_data['y']
file_ids_train = train_data['file_ids']
user_ids_train = train_data['user_ids']

test_data = np.load(os.path.join(KAGGLE_PATH, 'test_data.npz'), allow_pickle=True)
X_test_raw = test_data['X']
file_ids_test = test_data['file_ids']

## 2. Grandmaster Feature Extraction (Accel + Gyro getrennt)

In [ ]:
def extract_features_np(X):
    def calc_stats(arr, axis):
        f = []
        
        # 1. Basis-Stats für alle 6 Kanäle
        f.append(np.mean(arr, axis=axis))
        f.append(np.std(arr, axis=axis))
        f.append(np.min(arr, axis=axis))
        f.append(np.max(arr, axis=axis))
        f.append(np.median(arr, axis=axis))
        f.append(np.percentile(arr, 25, axis=axis))
        f.append(np.percentile(arr, 75, axis=axis))
        f.append(skew(arr, axis=axis))
        f.append(kurtosis(arr, axis=axis))
        f.append(np.max(arr, axis=axis) - np.min(arr, axis=axis))
        f.append(np.sum(arr**2, axis=axis))
        
        diffs = np.diff(arr, axis=axis)
        f.append(np.mean(np.abs(diffs), axis=axis))
        f.append(np.std(diffs, axis=axis))
        
        fft_vals = np.abs(np.fft.rfft(arr, axis=axis))
        f.append(np.mean(fft_vals, axis=axis))
        f.append(np.std(fft_vals, axis=axis))
        f.append(np.max(fft_vals, axis=axis))
        
        def calc_corr(a, b, ax):
            a_mean = np.mean(a, axis=ax, keepdims=True)
            b_mean = np.mean(b, axis=ax, keepdims=True)
            a_std = np.std(a, axis=ax, keepdims=True)
            b_std = np.std(b, axis=ax, keepdims=True)
            cov = np.mean((a - a_mean) * (b - b_mean), axis=ax, keepdims=True)
            return np.squeeze(cov / (a_std * b_std + 1e-8), axis=ax)
        
        # 2. Korrelationen Accelerometer (Kanäle 0, 1, 2)
        f.append(calc_corr(arr[..., 0], arr[..., 1], axis))
        f.append(calc_corr(arr[..., 0], arr[..., 2], axis))
        f.append(calc_corr(arr[..., 1], arr[..., 2], axis))
        
        # 3. Korrelationen Gyroscope (Kanäle 3, 4, 5)
        f.append(calc_corr(arr[..., 3], arr[..., 4], axis))
        f.append(calc_corr(arr[..., 3], arr[..., 5], axis))
        f.append(calc_corr(arr[..., 4], arr[..., 5], axis))
        
        # 4. Magnituden (Vektorlänge)
        mag_acc = np.sqrt(arr[..., 0]**2 + arr[..., 1]**2 + arr[..., 2]**2)
        mag_gyro = np.sqrt(arr[..., 3]**2 + arr[..., 4]**2 + arr[..., 5]**2)
        
        f.append(np.mean(mag_acc, axis=axis, keepdims=True))
        f.append(np.std(mag_acc, axis=axis, keepdims=True))
        f.append(np.max(mag_acc, axis=axis, keepdims=True))
        
        f.append(np.mean(mag_gyro, axis=axis, keepdims=True))
        f.append(np.std(mag_gyro, axis=axis, keepdims=True))
        f.append(np.max(mag_gyro, axis=axis, keepdims=True))
        
        # 5. Cross-Sensor Korrelation
        f.append(calc_corr(mag_acc, mag_gyro, axis))
        
        return f

    # Globale Features (300 Sekunden)
    global_feats = [f.reshape(X.shape[0], -1) for f in calc_stats(X, axis=1)]
    
    # Sub-Window Features: Diesmal 10 Blöcke à 30 Sekunden für extrem hohe Granularität!
    X_sub = X.reshape(X.shape[0], 10, 30, 6)
    sub_feats = [f.reshape(X.shape[0], -1) for f in calc_stats(X_sub, axis=2)]
    
    return np.concatenate(global_feats + sub_feats, axis=1)

print("Extrahiere Features...")
X_train_feat = extract_features_np(X_train_raw)
X_test_feat = extract_features_np(X_test_raw)
print(f"Anzahl der generierten Features pro Sample: {X_train_feat.shape[1]}")

## 3. LightGBM Training

In [ ]:
gkf = GroupKFold(n_splits=5)
models = []
scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_feat, y_train, groups=user_ids_train)):
    X_tr, X_va = X_train_feat[train_idx], X_train_feat[val_idx]
    y_tr, y_va = y_train[train_idx], y_train[val_idx]
    
    clf = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.005, # Noch feineres Lernen
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        subsample=0.8,
        colsample_bytree=0.5, # Nur 50% der Features pro Baum (hilft enorm gegen Overfitting bei 1000+ Features)
        max_depth=8,
        num_leaves=64,
        verbose=-1
    )
    
    clf.fit(
        X_tr, y_tr, 
        eval_set=[(X_va, y_va)], 
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    val_preds = clf.predict(X_va)
    fold_f1 = f1_score(y_va, val_preds, average='macro')
    scores.append(fold_f1)
    models.append(clf)
    
    print(f"Fold {fold+1} F1-Macro: {fold_f1:.4f}")

print(f"\nOverall Cross-Validation F1-Macro: {np.mean(scores):.4f}")

## 4. Test Predictions und Kaggle Submission

In [ ]:
test_preds_proba = np.zeros((len(X_test_feat), 6))

for clf in models:
    test_preds_proba += clf.predict_proba(X_test_feat) / len(models)

final_preds = np.argmax(test_preds_proba, axis=1)

submission = pd.DataFrame({
    'Id': file_ids_test,
    'Label': final_preds
})

submission.to_csv('submission_grandmaster.csv', index=False)
print("Saved submission_grandmaster.csv! Ready for Kaggle upload.")
submission.head()